# Generate Leetcode Preparation Templates
This notebook retrieves problem information and company question mapping statistics from Supabase, constructs structured preparation schedules (30-day, 60-day, 90-day templates) for the top 25 companies, and upserts them back to Supabase.

In [ ]:
import os
import json
from dotenv import load_dotenv
from supabase import create_client, Client
from collections import defaultdict
import math

In [ ]:
# Load environment variables
load_dotenv("../../apps/api/.env")

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY") or os.getenv("SUPABASE_SECRET_KEY") or os.getenv("SUPABASE_KEY") or os.getenv("NEXT_PUBLIC_SUPABASE_PUBLISHABLE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    print("Missing Supabase credentials. Make sure .env is set correctly.")
else:
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("Supabase Client initialized successfully.")

In [ ]:
# Define the logical sequence of topics for the templates
TOPIC_SEQUENCE = [
    {"name": "Arrays & Hashing", "tags": ["array", "hash-table"]},
    {"name": "Strings", "tags": ["string"]},
    {"name": "Two Pointers & Sliding Window", "tags": ["two-pointers", "sliding-window"]},
    {"name": "Linked Lists", "tags": ["linked-list"]},
    {"name": "Stacks & Queues", "tags": ["stack", "queue"]},
    {"name": "Trees", "tags": ["tree", "binary-tree", "binary-search-tree"]},
    {"name": "Heaps & Priority Queue", "tags": ["heap", "priority-queue"]},
    {"name": "Graphs", "tags": ["graph", "breadth-first-search", "depth-first-search"]},
    {"name": "Dynamic Programming", "tags": ["dynamic-programming", "memoization"]},
    {"name": "Bit Manipulation & Math", "tags": ["bit-manipulation", "math", "geometry"]},
    {"name": "System Design & Advanced", "tags": ["system-design", "design", "trie", "segment-tree"]}
]

In [ ]:
def fetch_data():
    print("Fetching problems and company mappings...")
    all_problems = {}
    limit = 1000
    offset = 0
    
    # Fetch all problems
    while True:
        res = supabase.table('lc_problems').select('id, title, slug, difficulty, topic_tags, ac_rate').range(offset, offset + limit - 1).execute()
        if not res.data:
            break
        for p in res.data:
            all_problems[p['id']] = p
        
        offset += limit
        if len(res.data) < limit:
            break
            
    print(f"Fetched {len(all_problems)} problems.")
    
    # Fetch company questions
    company_questions = defaultdict(list)
    company_counts = defaultdict(int)
    offset = 0
    
    while True:
        res = supabase.table('lc_company_questions').select('problem_id, company, frequency').range(offset, offset + limit - 1).execute()
        if not res.data:
            break
        for cq in res.data:
            company = cq['company'].lower().strip()
            pid = cq['problem_id']
            if pid in all_problems:
                company_questions[company].append({
                    "problem": all_problems[pid],
                    "frequency": cq['frequency']
                })
                company_counts[company] += 1
                
        offset += limit
        print(f"Fetched {offset} company mappings...", end='\r')
        if len(res.data) < limit:
            break
            
    print(f"\nFetched mappings for {len(company_questions)} companies.")
    return company_questions, company_counts

In [ ]:
def build_template(company, duration_days, questions_pool):
    weeks = math.ceil(duration_days / 7)
    topics_per_week = max(1, len(TOPIC_SEQUENCE) // weeks)
    template_weeks = []
    
    # Sort pool by frequency descending
    sorted_pool = sorted(questions_pool, key=lambda x: x['frequency'], reverse=True)
    used_problem_ids = set()
    total_questions = 0
    
    for w in range(weeks):
        week_topics = TOPIC_SEQUENCE[w * topics_per_week : (w + 1) * topics_per_week]
        if w == weeks - 1:
            week_topics = TOPIC_SEQUENCE[w * topics_per_week :]
            
        week_name = " & ".join([t['name'] for t in week_topics])
        week_tags = set()
        for t in week_topics:
            week_tags.update(t['tags'])
            
        week_questions = []
        target_q_count = {30: 15, 60: 10, 90: 7}.get(duration_days, 10)
        
        for q in sorted_pool:
            p = q['problem']
            if p['id'] in used_problem_ids:
                continue
                
            p_tags = set(p.get('topic_tags') or [])
            if p_tags.intersection(week_tags) or not week_tags:
                week_questions.append({
                    "id": p['id'],
                    "title": p['title'],
                    "slug": p['slug'],
                    "difficulty": p['difficulty']
                })
                used_problem_ids.add(p['id'])
                
            if len(week_questions) >= target_q_count:
                break
                
        # Fallback if there are not enough questions by tag
        if len(week_questions) < target_q_count:
            for q in sorted_pool:
                p = q['problem']
                if p['id'] not in used_problem_ids:
                    week_questions.append({
                        "id": p['id'],
                        "title": p['title'],
                        "slug": p['slug'],
                        "difficulty": p['difficulty']
                    })
                    used_problem_ids.add(p['id'])
                if len(week_questions) >= target_q_count:
                    break
                    
        easy = sum(1 for q in week_questions if q['difficulty'] == 'Easy')
        medium = sum(1 for q in week_questions if q['difficulty'] == 'Medium')
        hard = sum(1 for q in week_questions if q['difficulty'] == 'Hard')
        
        est_hours = round((easy * 20 + medium * 45 + hard * 90) / 60, 1)
        
        template_weeks.append({
            "week": w + 1,
            "theme": week_name,
            "questions": week_questions,
            "metrics": {
                "easy": easy,
                "medium": medium,
                "hard": hard,
                "estimated_hours": est_hours,
                "total": len(week_questions)
            }
        })
        total_questions += len(week_questions)
        
    return {
        "weeks": template_weeks,
        "total_weeks": weeks,
        "total_questions": total_questions
    }

### Fetch problem and company data

In [ ]:
company_questions, company_counts = fetch_data()

### Identify Top 25 Companies

In [ ]:
top_companies = sorted(company_counts.items(), key=lambda x: x[1], reverse=True)[:25]
print(f"Top 25 companies selected for template generation:")
for index, (company, count) in enumerate(top_companies):
    print(f"{index + 1}. {company.upper()}: {count} questions mapped")

### Generate Templates

In [ ]:
durations = [30, 60, 90]
templates_to_insert = []

for company, count in top_companies:
    pool = company_questions[company]
    for duration in durations:
        print(f"Building template for {company} - SDE - {duration} days...")
        template_data = build_template(company, duration, pool)
        template_id = f"{company.replace(' ', '_')}_sde_{duration}day"
        
        templates_to_insert.append({
            "id": template_id,
            "company": company,
            "role": "SDE",
            "duration_days": duration,
            "total_weeks": template_data["total_weeks"],
            "total_questions": template_data["total_questions"],
            "template_data": template_data,
            "generated_from_question_count": count
        })

print(f"Generated {len(templates_to_insert)} total templates in memory.")

### Upsert Templates to Supabase

In [ ]:
print(f"Upserting {len(templates_to_insert)} templates to Supabase...")

batch_size = 50
for i in range(0, len(templates_to_insert), batch_size):
    batch = templates_to_insert[i:i+batch_size]
    res = supabase.table('prep_templates').upsert(batch).execute()
    print(f"Upserted batch {i // batch_size + 1} of {math.ceil(len(templates_to_insert)/batch_size)}")
    
print("Done! All templates generated and saved successfully.")